Customer Churn Prediction: Week 1 Analysis
Project Phase: Setup & Data Ingestion

Date: April 3, 2026

1. Project Objective
The goal of this analysis is to perform Exploratory Data Analysis (EDA) and build a baseline machine learning model to predict customer churn using the Telco dataset. By identifying patterns in customer behavior, we aim to understand why customers leave and provide actionable insights.

2. Environment Setup
IDE: Visual Studio Code
Libraries Used: pandas, numpy, matplotlib, seaborn, and scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Telco-Customer-Churn.csv')
print(df.shape)
df.head(10)

print("--- Data Structure (.info) ---")
df.info()

print("\n--- Summary Statistics (.describe) ---")
display(df.describe())

print("\n--- Categorical Value Counts ---")
print(df['Contract'].value_counts())
print("\n", df['InternetService'].value_counts())

3. Initial Data Observations
After loading the dataset, here are the initial findings:

Dataset Shape: The dataset contains 7,043 rows and 21 columns.

Target Variable: Churn (Yes/No).

Data Inspection ObservationsData Types: Most features are object (categorical) or int64.

Missing Values: While .info() suggests no nulls, the .to_numeric conversion revealed 11 hidden missing values in TotalCharges.

Summary Statistics:
The average tenure is approximately 32 months.
MonthlyCharges vary significantly, with a mean around 64.76, suggesting a diverse range of service plans.

Categorical Distribution:
~ Contract: A large portion of the customer base is on a Month-to-month contract, which is a known high-risk factor for churn.
~Internet Service: Fiber optic is a major service category, which we later observed correlates with higher monthly costs

Key Features: Includes customer demographics (gender, seniority), services signed up for (Internet, Streaming, Tech Support), and financial data (Monthly Charges, Total Charges).

Immediate Note: The TotalCharges column is currently stored as an object type rather than float. This suggests the presence of empty strings or formatting issues that must be addressed during the cleaning phase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Telco-Customer-Churn.csv')

# 1. Convert TotalCharges to numeric (handles the blank strings)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Check how many null values were created
print(f"Missing values in TotalCharges: {df['TotalCharges'].isnull().sum()}")

# 3. Impute nulls with the median (as per project guidelines)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 4. Verify cleaning
print("Cleaned dtypes:")
print(df[['MonthlyCharges', 'TotalCharges']].dtypes)

from sklearn.preprocessing import LabelEncoder

# 1. Binary Encoding (Yes/No, Male/Female)
le = LabelEncoder()
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# 2. One-Hot Encoding (Multi-class columns like InternetService, Contract)
# This creates separate columns for each category (e.g., Contract_Month-to-month)
df = pd.get_dummies(df, columns=['MultipleLines', 'InternetService', 'OnlineSecurity', 
                                 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                                 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'])

# 3. Drop CustomerID as it's not a predictive feature
df.drop('customerID', axis=1, inplace=True)

print(f"New shape after encoding: {df.shape}")
df.head()



4. Data Cleaning & Preprocessing
Handling Missing Values: Identified 11 blank strings in TotalCharges. These were coerced to NaN and imputed using the median to maintain data integrity without removing rows.

Feature Encoding:

Used Label Encoding for binary features (e.g., Churn, Gender) to convert them into 0 and 1.

Applied One-Hot Encoding to multi-class categorical variables to avoid implying a numerical hierarchy between categories like "DSL" and "Fiber optic".

Feature Selection: Dropped customerID as it is a unique identifier and does not contribute to the model's predictive power.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Separate Features (X) and Target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# 2. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale Numeric Features
# We only scale 'tenure', 'MonthlyCharges', and 'TotalCharges'
scaler = StandardScaler()
cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Fit only on training data to prevent "Data Leakage"
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("\nClass balance in Training set:")
print(y_train.value_counts(normalize=True))

5. Feature Scaling & Train-Test Split

Train-Test Split: The data was split into 80% training and 20% testing sets. This allows us to evaluate the model on "unseen" data to ensure it generalizes well.

Feature Scaling: Used StandardScaler on numeric features (tenure, MonthlyCharges, TotalCharges).

Rationale: Standard scaling (Z-score normalization) was chosen because it centers the data around a mean of 0 and a standard deviation of 1.

Data Leakage Prevention: The scaler was fit only on the training data. The test data was then transformed using those training parameters. This ensures the model has no "future knowledge" of the test set distribution during training.

Class Balance: Observed that the dataset is imbalanced (approx. 26% churners). This means we should focus on Recall during evaluation rather than just Accuracy.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# visual style
sns.set_theme(style="whitegrid")

# figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Chart 1: Churn Distribution (Class Balance)
sns.countplot(x='Churn', data=df, ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Customer Churn')
axes[0].set_xticklabels(['Stayed (0)', 'Churned (1)'])

# Chart 2: Tenure vs Churn
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[1], palette='magma')
axes[1].set_title('Tenure (Months) by Churn Status')

# Chart 3: Monthly Charges vs Churn
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[2], palette='rocket')
axes[2].set_title('Monthly Charges by Churn Status')

plt.tight_layout()
plt.show()

# Chart 4: Correlation Heatmap (Top features only for clarity)
plt.figure(figsize=(12, 8))
top_corr = df.corr()['Churn'].sort_values(ascending=False).head(10)
sns.heatmap(df[top_corr.index].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Top 10 Features Correlated with Churn')
plt.show()

# Chart 5: Contract Type vs Churn (Critical Insight)
plt.figure(figsize=(8, 5))
# We use the original 'Contract' categories if available, or the dummy columns
sns.barplot(x=['Month-to-month', 'One year', 'Two year'], 
            y=[df['Contract_Month-to-month'].mean(), df['Contract_One year'].mean(), df['Contract_Two year'].mean()])
plt.title('Average Churn Rate by Contract Type')
plt.ylabel('Churn Probability')
plt.show()

6. Exploratory Data Analysis (EDA) - Part 1

Churn Distribution: The dataset is imbalanced, with approximately 26.5% of customers having churned. This informs us that Recall will be a more important metric than Accuracy, as we need to minimize "False Negatives" (missing a customer who is about to leave).

Tenure vs. Churn: The box plot reveals that customers who churn typically have a much shorter tenure (often less than 15-20 months) compared to loyal customers.

Monthly Charges: There is a clear trend showing that churned customers tend to have higher monthly charges. This suggests that high costs may be a primary driver for customer exit.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

# 2. Train Decision Tree (max_depth=5 as per instructions)
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_preds = dt_model.predict(X_test)

# 3. function to display clean results
def evaluate_model(name, y_true, y_pred):
    print(f"--- {name} Performance ---")
    print(classification_report(y_true, y_pred))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Run evaluation for both
evaluate_model("Logistic Regression", y_test, lr_preds)
evaluate_model("Decision Tree", y_test, dt_preds)

7. Model Training & Evaluation

Algorithms Selected: Implemented Logistic Regression as a baseline and a Decision Tree (max_depth=5) for non-linear pattern recognition. 

Metric Importance: While Accuracy tells us how many total predictions were correct, Recall is our primary focus. Catching a customer who is about to churn (True Positive) is more valuable than correctly identifying those who stay.

8. Final Conclusions & Recommendations
Key Findings from EDA:

High Risk Factors: Customers on Month-to-Month contracts with Fiber Optic internet and High Monthly Charges are the most likely to churn.

Tenure Impact: New customers (low tenure) are at a much higher risk of leaving than long-term customers.

Model Comparison:
Logistic Regression: Achieved 82% Accuracy. It is more precise but misses more actual churners (lower Recall).

Decision Tree (max_depth=5): Achieved 80% Accuracy but provided a higher Recall of 64% for churners.

Final Recommendation:
I recommend the Decision Tree model for this specific business case.

Rationale: In churn prediction, the cost of "missing" a customer who is about to leave is much higher than the cost of accidentally sending a discount to someone who was going to stay. The Decision Tree’s superior Recall makes it the better tool for proactive customer retention.

Limitations: The model is trained on a relatively small imbalanced dataset. Future improvements could include using SMOTE for oversampling or exploring ensemble methods like Random Forest.